In [1]:
import time
import json
import os
import traceback
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Layout, jslink, IntRangeSlider
from IPython.display import display, clear_output
import zarr
import dask.array as da
from dask_image.ndfilters import gaussian_filter as gaussian_filter_dask
from scipy import ndimage as ndi
from skimage import morphology
import tifffile

from masking import get_mask
from estimate_background import background_estimation

class ZarrMaskingApp:
    """
    Interactive Jupyter widget app for 3D mask generation from Zarr Spim data.
    """
    # --- Configurable constants ---
    DEFAULT_ZARR_PATH = 's3://aind-open-data/exaSPIM_754615_2025-01-23_16-44-53/SPIM.ome.zarr'
    RAW_SAVE_DIR = "/results/raw"
    MASK_SAVE_DIR = "/results/mask"
    TIFF_COMPRESSION = "zlib"

    def __init__(self):
        # State variables
        self.threshold_results = {}
        self.current_array_name = None
        self.current_zarr_array = None  # Corrected numpy array, shape (z, y, x)
        self.zarr_group = None
        self.current_calculated_mask = None
        self.dataset_name = None

        # Build UI
        self._create_widgets()
        self._link_callbacks()
        self._arrange_layout()
        self._update_save_tiff_button_state()
        display(self.ui)

    # ---- Widget Creation ----

    def _create_widgets(self):
        # Zarr path and load group
        self.zarr_path_input = widgets.Text(
            value=self.DEFAULT_ZARR_PATH, placeholder='Enter path to Zarr Group',
            description='Zarr Group Path:', style={'description_width': 'initial'}, layout=Layout(width='80%')
        )
        self.load_group_button = widgets.Button(description="Load Group")
        self.array_selector = widgets.Dropdown(
            options=['Load Group First'], description='Array/Tile:',
            style={'description_width': 'initial'}, disabled=True
        )
        self.slice_slider = widgets.IntSlider(
            min=0, max=10, step=1, value=0, description='View Slice:', continuous_update=True,
            style={'description_width': 'initial'}, layout=Layout(width='80%'), disabled=True
        )
        self.mask_slice_range_slider = IntRangeSlider(
            value=[0, 10], min=0, max=10, step=1,
            description='Mask Range (Z):', disabled=True,
            continuous_update=False, orientation='horizontal',
            readout=True, readout_format='d',
            style={'description_width': 'initial'}, layout=Layout(width='80%')
        )
        self.FINE_THRESHOLD_STEP = 1
        self.threshold_slider = widgets.FloatSlider(
            min=0, max=5000, step=1, value=50, description='Threshold:', continuous_update=True, readout=False,
            style={'description_width': 'initial'}, layout=Layout(width='70%'), disabled=True
        )
        self.threshold_fine_control = widgets.BoundedFloatText(
            min=0, max=5000, step=self.FINE_THRESHOLD_STEP, value=50, description='',
            layout=Layout(width='30%'), disabled=True
        )
        jslink((self.threshold_slider, 'value'), (self.threshold_fine_control, 'value'))
        self.threshold_controls = HBox([self.threshold_slider, self.threshold_fine_control])

        self.calculate_mask_button = widgets.Button(
            description="Calculate Mask (Selected Range)",
            button_style='info',
            tooltip='Calculate the 3D mask within the selected slice range using the current threshold',
            icon='calculator', disabled=True
        )
        self.load_mask_path_input = widgets.Text(
            value='', placeholder='Enter mask file path',
            description='Mask File:', style={'description_width': 'initial'}, layout=Layout(width='60%')
        )
        self.load_mask_button = widgets.Button(
            description='Load Mask from Disk', button_style='warning',
            tooltip='Load a mask file and display as current mask',
            icon='upload'
        )

        self.plot_output = widgets.Output()
        self.status_output = widgets.Output()

        self.save_filename_input = widgets.Text(
            value='group_thresholds.json', placeholder='Enter filename',
            description='Save JSON Filename:', style={'description_width': 'initial'}, layout=Layout(width='40%')
        )
        self.save_button = widgets.Button(
            description='Save Thresholds', button_style='success',
            tooltip='Save thresholds to JSON', icon='save'
        )
        self.save_tiff_button = widgets.Button(
            description="Save Data & Mask TIFF", button_style='primary',
            tooltip='Save data and calculated mask. Load data and calculate mask first.',
            icon='picture-o', disabled=True
        )
        # Mask processing tools
        self.postprocess_label = widgets.Label("Post-process Mask:")
        self.dilate_button = widgets.Button(description="Dilate", tooltip="Dilate mask (expand regions)")
        self.erode_button = widgets.Button(description="Erode", tooltip="Erode mask (shrink regions)")
        self.fill_button = widgets.Button(description="Fill Holes", tooltip="Fill holes in mask")
        self.size_filter_button = widgets.Button(description="Keep Large Objects", tooltip="Remove small objects from mask")
        self.size_filter_input = widgets.IntText(value=100, description="Min Size:", layout=Layout(width="150px"))
        self.morph_radius_input = widgets.BoundedIntText(
            value=1, min=1, max=10, step=1, description="Radius:",
            layout=Layout(width="120px")
        )
        self.postprocess_controls = HBox([
            self.morph_radius_input,
            self.dilate_button, 
            self.erode_button, 
            self.fill_button, 
            self.size_filter_input, 
            self.size_filter_button
        ])



    def _arrange_layout(self):
        self.group_controls = HBox([self.zarr_path_input, self.load_group_button])
        
        self.save_controls = HBox(
            [self.save_filename_input, self.save_button, self.save_tiff_button],
            layout=Layout(justify_content='flex-start')
        )
        self.load_mask_controls = HBox([self.load_mask_path_input, self.load_mask_button])

        self.array_controls = VBox([
            self.array_selector,
            self.slice_slider,
            self.mask_slice_range_slider,
            self.threshold_controls,
            self.calculate_mask_button,
            self.postprocess_label,
            self.postprocess_controls,
        ])
        
        self.ui = VBox([
            widgets.HTML("<h2>Zarr Array Masking Tool</h2>"),
            self.group_controls,
            widgets.HTML("<hr>"),
            self.array_controls,
            self.load_mask_controls, 
            widgets.HTML("<hr>"),
            self.save_controls,
            widgets.HTML("<hr>"),
            self.plot_output,
            self.status_output
        ])



    # ---- Callback/Logic Wiring ----

    def _link_callbacks(self):
        self.load_group_button.on_click(self.open_group_and_populate_dropdown)
        self.array_selector.observe(self.on_array_change, names='value')
        self.slice_slider.observe(self.update_plot, names='value')
        self.threshold_slider.observe(self.update_plot, names='value')
        self.save_button.on_click(self.on_save_button_clicked)
        self.calculate_mask_button.on_click(self.on_calculate_mask_clicked)
        self.save_tiff_button.on_click(self.on_save_tiff_clicked)
        self.load_mask_button.on_click(self.on_load_mask_clicked)
        self.dilate_button.on_click(self.on_dilate_clicked)
        self.erode_button.on_click(self.on_erode_clicked)
        self.fill_button.on_click(self.on_fill_holes_clicked)
        self.size_filter_button.on_click(self.on_size_filter_clicked)


    # ---- UI Utility ----

    def show_status(self, msg: str, clear=True):
        with self.status_output:
            if clear: clear_output(wait=True)
            print(msg)

    def _update_save_tiff_button_state(self):
        can_save = self.current_zarr_array is not None and self.current_calculated_mask is not None
        self.save_tiff_button.disabled = not can_save
        if can_save and self.current_array_name:
            base_name = self.current_array_name.replace('/', '_').replace('\\', '_')
            self.save_tiff_button.tooltip = f"Save {base_name} data & mask to TIFFs"
        elif self.current_zarr_array is None:
            self.save_tiff_button.tooltip = "Load data to enable."
        elif self.current_calculated_mask is None:
            self.save_tiff_button.tooltip = "Calculate mask to enable."
        else:
            self.save_tiff_button.tooltip = "Load data & mask to enable."

    def _disable_threshold_widgets(self, disabled: bool):
        self.threshold_slider.disabled = disabled
        self.threshold_fine_control.disabled = disabled
        self.calculate_mask_button.disabled = disabled

    def _disable_controls_after_error(self, clear_plot=False, disable_selector=False):
        if clear_plot: self.plot_output.clear_output()
        self.slice_slider.disabled = True
        self.mask_slice_range_slider.disabled = True
        self._disable_threshold_widgets(True)
        if disable_selector:
            self.array_selector.disabled = True
        self.save_tiff_button.disabled = True
        self.save_tiff_button.tooltip = "Unavailable due to error."

    # ---- Main Functional Logic ----

    def open_group_and_populate_dropdown(self, b=None, res=4, ch="488"):
        path = self.zarr_path_input.value.strip()
        self.dataset_name = Path(path).parent.name
        self.show_status(f"Attempting to open Zarr group: {path}...")
        try:
            self.zarr_group = zarr.open_group(path, mode='r')
            array_keys = []
            # Search logic for FOV/channel_res structure (most common)
            for k_group in self.zarr_group.keys():
                if isinstance(self.zarr_group[k_group], zarr.Group):
                    for k_channel_res in self.zarr_group[k_group].keys():
                        if ch in k_channel_res and f"res_{res}" in k_channel_res:
                            potential_array_path = f"{k_group}/{k_channel_res}"
                            if potential_array_path in self.zarr_group and isinstance(self.zarr_group[potential_array_path], zarr.Array):
                                array_keys.append(potential_array_path)
            # Fallback: Try original logic
            if not array_keys:
                for k in self.zarr_group.keys():
                    if ch in k and f"{k}/{res}" in self.zarr_group and isinstance(self.zarr_group[f"{k}/{res}"], zarr.Array):
                        array_keys.append(f"{k}/{res}")
            if not array_keys:
                self.show_status("Warning: No suitable Zarr arrays found...")
                self.array_selector.options = ['No arrays found']
                self.array_selector.disabled = True
                self._disable_controls_after_error(clear_plot=True)
            else:
                self.show_status(f"Group opened. Found matching arrays: {array_keys}")
                self.array_selector.unobserve(self.on_array_change, names='value')
                self.array_selector.options = sorted(array_keys)
                self.array_selector.disabled = False
                self.array_selector.observe(self.on_array_change, names='value')
                self.current_calculated_mask = None
                self.calculate_mask_button.disabled = True
                self.mask_slice_range_slider.disabled = True
                self._update_save_tiff_button_state()
        except ImportError:
            self.show_status("ERROR: Missing S3 library (e.g., s3fs)\n" + traceback.format_exc())
            self._disable_controls_after_error(clear_plot=True)
        except Exception as e:
            self.show_status(f"ERROR opening Zarr: {path}\n" + traceback.format_exc())
            self._disable_controls_after_error(clear_plot=True, disable_selector=True)

    def on_array_change(self, change):
        new_array_name = change.get('new')
        self.show_status(f"Array selection changed to: {new_array_name}")
        self.current_calculated_mask = None
        self.calculate_mask_button.disabled = True
        self.mask_slice_range_slider.disabled = True
        self._update_save_tiff_button_state()
        if new_array_name and self.zarr_group is not None:
            if new_array_name in self.array_selector.options and new_array_name not in ['No arrays found', 'Load Group First']:
                if new_array_name in self.zarr_group and isinstance(self.zarr_group[new_array_name], zarr.Array):
                    self.load_and_update_for_array(new_array_name)
                else:
                    self.show_status(f"Selected item '{new_array_name}' is not a loadable Zarr array.")
                    self._disable_controls_after_error(clear_plot=True)
            elif new_array_name not in ['No arrays found', 'Load Group First']:
                self.show_status(f"Selected array '{new_array_name}' not found or group not loaded.")
                self._disable_controls_after_error(clear_plot=True)

    def load_and_update_for_array(self, array_name: str):
        self.show_status(f"\nLoading array: {array_name}...")
        if self.zarr_group is None:
            self.show_status("Error: Zarr group not open.")
            return
        try:
            zarr_array_proxy = self.zarr_group[array_name]
            self.show_status(f"Accessing {array_name}, shape={zarr_array_proxy.shape}, dtype={zarr_array_proxy.dtype}...\nLoading data & processing...")
            t_start = time.time()
            temp_numpy_array = da.from_array(zarr_array_proxy).squeeze()
            temp_numpy_array = gaussian_filter_dask(temp_numpy_array, sigma=2).compute()
            t_end = time.time()
            self.show_status(f"Data loaded & filtered ({t_end - t_start:.2f}s). Estimating background...")
            bkg = background_estimation(temp_numpy_array)
            corrected_array_float = temp_numpy_array.astype(np.float32) - bkg
            original_dtype = zarr_array_proxy.dtype if hasattr(zarr_array_proxy, 'dtype') else np.uint16
            corrected_array = np.clip(
                corrected_array_float, 0,
                np.iinfo(original_dtype).max if np.issubdtype(original_dtype, np.integer) else 65535.0
            ).astype(original_dtype)
            self.current_zarr_array = corrected_array
            self.current_array_name = array_name
            self.current_calculated_mask = None
            self._update_save_tiff_button_state()
            if self.current_zarr_array.ndim != 3:
                raise ValueError(f"Expected 3D array, got {self.current_zarr_array.ndim}D.")
            num_slices = self.current_zarr_array.shape[0]
            # Slice sliders
            self.slice_slider.max = num_slices - 1 if num_slices > 0 else 0
            self.slice_slider.value = min(self.slice_slider.value, self.slice_slider.max)
            if self.slice_slider.value < 0 and num_slices > 0:
                self.slice_slider.value = 0
            self.slice_slider.disabled = (num_slices <= 0)
            self.mask_slice_range_slider.max = num_slices - 1 if num_slices > 0 else 0
            self.mask_slice_range_slider.value = [0, self.mask_slice_range_slider.max]
            self.mask_slice_range_slider.disabled = (num_slices <= 0)
            # Threshold widgets
            data_min_val = 0.0
            data_max_val = float(np.max(self.current_zarr_array)) if num_slices > 0 and np.any(self.current_zarr_array) else (
                np.iinfo(self.current_zarr_array.dtype).max if np.issubdtype(self.current_zarr_array.dtype, np.integer) else 65535.0)
            if data_max_val < 1.0: data_max_val = 1.0
            self.threshold_slider.min = data_min_val
            self.threshold_slider.max = data_max_val
            self.threshold_slider.step = max(1.0, (data_max_val - data_min_val) / 5000)
            self.threshold_fine_control.min = data_min_val
            self.threshold_fine_control.max = data_max_val
            self.threshold_fine_control.step = max(0.1, (data_max_val - data_min_val) / 10000)
            if array_name in self.threshold_results:
                self.threshold_slider.value = self.threshold_results[array_name]
            else:
                initial_threshold_guess = float(np.percentile(self.current_zarr_array[self.current_zarr_array > 0], 50)) if num_slices > 0 and np.any(self.current_zarr_array > 0) else data_max_val / 2.0
                self.threshold_slider.value = initial_threshold_guess
                self.threshold_results[array_name] = initial_threshold_guess
            self._disable_threshold_widgets(num_slices <= 0)
            self.calculate_mask_button.disabled = (num_slices <= 0)
            self.show_status(f"Loaded '{array_name}'. Shape: {self.current_zarr_array.shape}, Range: [{np.min(self.current_zarr_array)}, {np.max(self.current_zarr_array)}]")
            self.update_plot(None)
        except KeyError:
            self.show_status(f"Error: Array '{array_name}' not found.")
            self._disable_controls_after_error(clear_plot=True)
        except MemoryError:
            self.show_status(f"ERROR: MemoryError loading {array_name}")
            self._disable_controls_after_error(clear_plot=True)
        except Exception as e:
            self.show_status(f"ERROR loading {array_name}:\n" + traceback.format_exc())
            self._disable_controls_after_error(clear_plot=True)
        finally:
            self._update_save_tiff_button_state()

    def on_calculate_mask_clicked(self, b):
        self.show_status("\n'Calculate Mask (Selected Range)' button clicked.")
        if self.current_zarr_array is None or self.current_array_name is None:
            self.show_status("Error: No array loaded to calculate mask for.", clear=False)
            return
        threshold = self.threshold_slider.value
        start_slice, end_slice = self.mask_slice_range_slider.value
        actual_end_slice = end_slice + 1
        if start_slice >= actual_end_slice:
            self.show_status(f"Warning: Start slice ({start_slice}) is not before end slice ({end_slice}). No mask will be calculated.", clear=False)
            self.current_calculated_mask = np.zeros_like(self.current_zarr_array, dtype=np.uint8)
            self.update_plot(None)
            self._update_save_tiff_button_state()
            return
        self.show_status(f"Calculating mask for '{self.current_array_name}' slices {start_slice}-{end_slice} with threshold {threshold:.2f}...", clear=False)
        self.calculate_mask_button.disabled = True
        self.save_tiff_button.disabled = True
        try:
            if self.current_calculated_mask is not None:
                k0, k1 = start_slice, actual_end_slice  
                self.current_calculated_mask[:k0,  :, :] = 0
                self.current_calculated_mask[k1:,  :, :] = 0
            else:
                sub_volume = self.current_zarr_array[start_slice:actual_end_slice, :, :]
                if sub_volume.size == 0:
                    self.show_status("Warning: Selected slice range is empty. No mask calculated.", clear=False)
                    mask_for_sub_volume = np.array([], dtype=np.uint8)
                else:
                    mask_for_sub_volume = get_mask(sub_volume, threshold)
                self.current_calculated_mask = np.zeros_like(self.current_zarr_array, dtype=np.uint8)
                if mask_for_sub_volume.size > 0 and mask_for_sub_volume.shape == sub_volume.shape:
                    self.current_calculated_mask[start_slice:actual_end_slice, :, :] = mask_for_sub_volume
                elif mask_for_sub_volume.size > 0:
                    self.show_status(f"Warning: Shape mismatch between sub-volume {sub_volume.shape} and its mask {mask_for_sub_volume.shape}. Mask not applied.", clear=False)
            self.show_status(f"Mask calculation finished. Full mask shape: {self.current_calculated_mask.shape}, processed slices: {start_slice}-{end_slice}", clear=False)
            self.update_plot(None)
        except Exception as e:
            self.show_status(f"ERROR during mask calculation:\n{traceback.format_exc()}")
            self.current_calculated_mask = None
        finally:
            self.calculate_mask_button.disabled = (self.current_zarr_array is None)
            self._update_save_tiff_button_state()

    def update_plot(self, change):
        if self.current_zarr_array is None or self.current_array_name is None:
            return
        slice_idx = self.slice_slider.value
        current_threshold = self.threshold_slider.value
        # Update threshold state
        if self.current_array_name is not None:
            self.threshold_results[self.current_array_name] = current_threshold
        try:
            original_slice = self.current_zarr_array[slice_idx, :, :]
            dynamic_mask = (original_slice > current_threshold).astype(np.uint8)
            calculated_mask_slice = None
            if self.current_calculated_mask is not None:
                if 0 <= slice_idx < self.current_calculated_mask.shape[0]:
                    calculated_mask_slice = self.current_calculated_mask[slice_idx, :, :]
            self._plot_current_slice_and_mask(
                original_slice, dynamic_mask, calculated_mask_slice,
                slice_idx, current_threshold, self.current_array_name
            )
        except IndexError:
            self.show_status(f"Error: Slice index {slice_idx} out of bounds for current array.")
            self.plot_output.clear_output()
        except Exception as e:
            self.show_status(f"ERROR generating plot:\n{traceback.format_exc()}")
            self.plot_output.clear_output()

    def _plot_current_slice_and_mask(self, image_slice, dynamic_mask_slice, calculated_mask_slice, slice_idx, threshold, array_name):
        with self.plot_output:
            clear_output(wait=True)
            num_subplots = 3 if calculated_mask_slice is not None else 2
            fig, axes = plt.subplots(1, num_subplots, figsize=(5 * num_subplots + 1, 5))
            if num_subplots == 1:
                axes = [axes]
            elif num_subplots == 2:
                axes = list(axes)
            vmin, vmax = 0, 100
            axes[0].imshow(image_slice, cmap='gray', vmin=vmin, vmax=vmax)
            axes[0].set_title(f'{array_name} - Slice {slice_idx}')
            axes[0].axis('off')
            axes[1].imshow(dynamic_mask_slice, cmap='gray', vmin=0, vmax=1)
            axes[1].set_title(f'Live Mask (>{threshold:.2f})')
            axes[1].axis('off')
            if calculated_mask_slice is not None:
                axes[2].imshow(calculated_mask_slice, cmap='gray', vmin=0, vmax=1)
                axes[2].set_title(f'Calculated Mask Sl {slice_idx}')
                axes[2].axis('off')
            plt.tight_layout()
            plt.show()

    def on_save_button_clicked(self, b):
        filename = self.save_filename_input.value.strip()
        self.show_status(f"Save JSON button clicked.")
        if not filename:
            self.show_status("Error: Please enter a filename for JSON thresholds.")
            return
        if not filename.lower().endswith('.json'):
            filename += '.json'
            self.save_filename_input.value = filename
        self.show_status(f"Attempting to save {len(self.threshold_results)} thresholds to '{filename}'...")
        try:
            serializable_results = {k: float(v) for k, v in self.threshold_results.items()}
            with open(filename, 'w') as f:
                json.dump(serializable_results, f, indent=4, sort_keys=True)
            self.show_status(f"Successfully saved {len(serializable_results)} thresholds to '{filename}'")
        except Exception as e:
            self.show_status(f"ERROR saving JSON file '{filename}':\n{traceback.format_exc()}")

    def on_save_tiff_clicked(self, b):
        self.show_status("\n'Save Data & Mask TIFF' button clicked.")
        if self.current_zarr_array is None:
            self.show_status("Error: No data array loaded.", clear=False)
            return
        if self.current_calculated_mask is None:
            self.show_status("Error: No mask calculated.", clear=False)
            return
        if self.current_array_name is None:
            self.show_status("Error: Array name unknown.", clear=False)
            return
        try:
            safe_name = self.current_array_name.replace('/', '_').replace('\\', '_').replace(':', '_')
            os.makedirs(self.RAW_SAVE_DIR, exist_ok=True)
            os.makedirs(self.MASK_SAVE_DIR, exist_ok=True)
            raw_out = os.path.join(self.RAW_SAVE_DIR, f"{self.dataset_name}_{safe_name}_data.tif")
            mask_out = os.path.join(self.MASK_SAVE_DIR, f"{self.dataset_name}_{safe_name}_mask.tif")
            self.show_status(f"Saving raw to '{raw_out}', mask to '{mask_out}'...", clear=False)
            mask_to_save = self.current_calculated_mask.astype(np.uint8)
            if self.current_zarr_array.shape != mask_to_save.shape:
                self.show_status(f"ERROR: Data shape {self.current_zarr_array.shape} != mask shape {mask_to_save.shape}!", clear=False)
                return
            self.save_tiff_button.disabled = True
            self.save_tiff_button.description = "Saving..."
            tifffile.imwrite(raw_out, self.current_zarr_array, imagej=True, compression=self.TIFF_COMPRESSION)
            self.show_status(f"Saved raw: {self.current_zarr_array.shape}, dtype {self.current_zarr_array.dtype}", clear=False)
            tifffile.imwrite(mask_out, mask_to_save, imagej=True, compression=self.TIFF_COMPRESSION)
            self.show_status(f"Saved mask: {mask_to_save.shape}, dtype {mask_to_save.dtype}", clear=False)
        except Exception as e:
            self.show_status(f"ERROR saving TIFFs:\n{traceback.format_exc()}")
        finally:
            self._update_save_tiff_button_state()
            self.save_tiff_button.description = "Save Data & Mask TIFF"

    def on_load_mask_clicked(self, b):
        mask_path = self.load_mask_path_input.value.strip()
        self.show_status(f"Attempting to load mask from: {mask_path}")
        if not mask_path or not os.path.isfile(mask_path):
            self.show_status("Mask file path is empty or does not exist.")
            return
        try:
            # Support both TIFF and NPY formats
            if mask_path.lower().endswith('.npy'):
                loaded_mask = np.load(mask_path)
            else:
                loaded_mask = tifffile.imread(mask_path)
            if self.current_zarr_array is None:
                self.show_status("Error: Load data before loading a mask!")
                return
            if loaded_mask.shape != self.current_zarr_array.shape:
                self.show_status(f"Error: Mask shape {loaded_mask.shape} does not match data shape {self.current_zarr_array.shape}.")
                return
            # Optional: convert dtype
            loaded_mask = (loaded_mask > 0).astype(np.uint8)
            self.current_calculated_mask = loaded_mask
            self.show_status(f"Mask loaded successfully from '{mask_path}'.")
            self.update_plot(None)
            self._update_save_tiff_button_state()
        except Exception as e:
            self.show_status(f"ERROR loading mask:\n{traceback.format_exc()}")

    def _apply_mask_operation(self, op_func, op_name):
        if self.current_calculated_mask is None:
            self.show_status(f"Error: No mask to {op_name}.", clear=False)
            return
        try:
            before = self.current_calculated_mask.copy()
            self.current_calculated_mask = op_func(self.current_calculated_mask)
            self.show_status(f"{op_name} operation applied to mask.")
            self.update_plot(None)
            self._update_save_tiff_button_state()
        except Exception as e:
            self.show_status(f"Error during {op_name}:\n{traceback.format_exc()}")
    
    def on_dilate_clicked(self, b):
        radius = self.morph_radius_input.value
        self._apply_mask_operation(
            lambda mask: morphology.binary_dilation(mask, morphology.ball(radius)).astype(np.uint8),
            f"Dilation (radius={radius})"
        )
    
    def on_erode_clicked(self, b):
        radius = self.morph_radius_input.value
        self._apply_mask_operation(
            lambda mask: morphology.binary_erosion(mask, morphology.ball(radius)).astype(np.uint8),
            f"Erosion (radius={radius})"
        )

    def on_fill_holes_clicked(self, b):
        self._apply_mask_operation(
            lambda mask: ndi.binary_fill_holes(mask).astype(np.uint8),
            "Fill Holes"
        )
    
    def on_size_filter_clicked(self, b):
        min_size = self.size_filter_input.value
        self._apply_mask_operation(
            lambda mask: morphology.remove_small_objects(mask.astype(bool), min_size=min_size).astype(np.uint8),
            f"Keep Objects >={min_size} voxels"
        )


app = ZarrMaskingApp()